# 第10回 演習：総合演習——フローを1周する（解答例・教員用）

## 今日の分析目標

**くちばしと体の計測値から、ペンギンの種類を当てたい。**

この演習が、前半戦の総まとめです。南極パーマー基地の3種のペンギン（Adelie / Chinstrap / Gentoo）のデータを主役に、分析のフローを**自分の手で一周**します。①データを理解して目標を具体化し、②前処理から評価までを組み、③結果を解釈して目標に答える——これまで一つずつ学んだ道具（分割・補完・ワンホット・交差検証・分類・Pipeline・GridSearchCV）を、総動員してください。TODOは6か所。各TODOにはヒントがあります。

出典：palmerpenguins（Horst, Hill & Gorman 2020、CC0）

### 深掘り：この回の歩き方——なぜ「目標 → 理解 → 前処理 → 評価 → 報告」の順に回すのか

この演習は、6つのTODOがバラバラの練習問題ではありません。**ぜんぶで一つの流れ**——分析の一周をなぞる、ひと続きの物語です。第3回から第9回までは、地図のどこか一か所だけを取り出して練習してきました。今日はその部品を、**目標設定 → データ理解 → 前処理 → 学習 → 評価 → 解釈・報告**という順に、通しでつなぎます。この節ではまず「なぜこの順なのか」を先にほどいておきます。各TODOが、この一本の背骨のどの関節にあたるのかを意識しながら進めてください。

**順番は飾りではなく、依存関係です。** 一つ前の段が決まらないと次の段が設計できない——それがこの並びの正体です。

- **なぜ目標が先頭か**：当てたいのは `species`（3種のどれか）だと決めて初めて、これが「**多クラス分類**」だと型が定まります。型が決まって初めて、使う手法（ロジスティック回帰）も、成績の測り方（正解率）も選べます。目標が曖昧なままモデルを回すのは、行き先を決めずに電車に乗るようなものです。
- **なぜ次がデータ理解か**：前処理の設計は、データの中身を見てからでないと書けません。どの列が数値でどの列がカテゴリか、欠損はどこにいくつあるか——これを知らずに補完やワンホットは組めません（第4回）。
- **なぜ前処理 → 学習 → 評価の順が動かせないか**：テストを最初に取り分け（第3回）、前処理は訓練データからだけ学ばせ（第4回・第9回）、実力は訓練の中の交差検証で測り（第5回）、最後にテストで一度だけ本番評価する。この順を崩すと**リーク**（答えの先読み）が起き、成績が水増しされます。
- **なぜ解釈・報告が最後か**：数字が出てからでないと解釈できませんし、解釈して初めて「目標に答えられたか」を言えます（第6回）。

各TODOと道具の対応も、この背骨に沿っています——①欠損の把握（第4回）、②`ColumnTransformer` で列ごとの前処理（第4回・第9回）、③`Pipeline` で一本化（第9回）、④交差検証（第5回）、⑤`GridSearchCV` とテスト評価・混同行列（第8回・第9回）、⑥係数の解釈（第6回）。今日のデータは 344 行 7 列、1 行が 1 匹のペンギンです。この小さな表を相手に、上の一周を最後まで自分の手で回しきる——それがこの回の主役です。以降の各節の深掘りでは、その関節ごとに「**なぜこの段がここに来るのか**」を、実際に出てくる数字とともに確かめていきます。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ

# データを読み込み、手がかり（特徴量）と正解（ラベル）に分け、訓練とテストに分割する
# ここから先、テストは最後の評価まで触らない（第3回）
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/penguins.csv')
num = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
cat = ['island', 'sex']
X = df[num + cat]
y = df['species']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('形:', df.shape)
print('訓練:', X_tr.shape, ' テスト:', X_te.shape)
df.head()

## 1. データを理解する——欠損を把握する

実データには、欠損がつきものです。前処理の方針を決める前に、**どの列に、どれだけ欠損があるか**を確かめましょう（第4回）。ここが分かっていないと、あとで補完の設計ができません。

### TODO①：列ごとの欠損数と、その合計を求める

### 深掘り：なぜ前処理より先に「データを理解する」のか——設計は観察のあとにしか書けない

いきなり `ColumnTransformer` を書きたくなるのを、ぐっとこらえて、まず数えるのがこの節です。**前処理の設計図は、データの実態を見てからでないと一行も書けない**——これが「理解が前処理より先に来る」理由です。

TODO① で欠損を数えると、次のことが分かります（TODO①を正しく解いたときの実行値）。計測4列（`bill_length_mm`・`bill_depth_mm`・`flipper_length_mm`・`body_mass_g`）はそれぞれ **2 個ずつ**、カテゴリの `sex` は **11 個**、合わせて **19 セル**が欠けています。この一枚の観察が、次節の設計をまるごと決めます——

- 欠損が**ある**と分かったから、補完（`SimpleImputer`）を前処理に入れる。もし 0 セルなら、そもそも補完器は要りません。
- 欠損が**数値とカテゴリの両方**にあると分かったから、数値は中央値・カテゴリは最頻値と、列の種類で補い方を変える。
- `sex` に欠損が集中していると分かったから、カテゴリ側の補完を軽視できない、と気づけます。

**捨てずに埋める理由も、数を見たから言える**　欠損のある行を丸ごと落とせば話は早いですが、19 セルを含む行を捨てると十数匹分の貴重な観測を失います。実データは一匹ずつ人が測った記録で、簡単には増やせません。だから「行を捨てる」より「値を埋める」を選ぶ——この判断も、欠損の量を**見て**初めて下せます。

**種の偏りを確かめ、stratify の妥当性を後追いで裏づける**　種の数は Adelie 152・Gentoo 124・Chinstrap 68 と、いちばん多い種と少ない種で 2 倍以上の開きがあります。極端ではありませんが無視もできない偏りです。じつは冒頭のセットアップでは、最初から `train_test_split(..., stratify=y)` と**比率を保つ分割**を済ませてあります（第3回）。ここで偏りを目で確かめると、その判断が正しかったことが**後追いで納得できます**——少数派の Chinstrap が偶然テストに偏ると評価が歪むので、比率をそろえて分けておくのが正解だった、というわけです。観察が先で分割が後、ではなく、**先に敷いておいた分割の意味を、観察で確かめて腑に落とす**——これも「理解」の大切な一面です。

**つまずき：観察を飛ばす分析**　EDA（探索的データ分析）を飛ばして前処理へ突っ込むと、欠損でモデルが止まる、カテゴリを数値と誤って標準化してしまう、偏りに気づかず評価がぶれる——といった手戻りが必ず起きます。**最初に自分の目でデータを眺める**のは、遠回りではなく最短路です。列の型と欠損の地図を手にして、次の設計へ進みましょう。


In [ ]:
# TODO: 各列に欠損がいくつあるかを数え、さらに全体の欠損セル数の合計も求めてください

# 解答例①：列ごとの欠損数と合計
missing = df.isnull().sum()
print(missing)
print('欠損セルの合計:', int(missing.sum()))
# → 計測値は各2、sex は11。合計19セル。カテゴリ側の sex に欠損が多い


## 2. 前処理を設計する——ColumnTransformer

数値の列とカテゴリの列では、必要な処理が違います。**数値**は「中央値で補完 → 標準化」、**カテゴリ**は「最頻値で補完 → ワンホット」。この列ごとの処理を、`ColumnTransformer` で一つに束ねます（第4回・第9回）。

### TODO②：数値とカテゴリで別々の前処理を組み、ColumnTransformer で束ねる

### 深掘り：なぜ列ごとに、そしてなぜ「補完 → 変換」の順なのか

理解した中身を、いよいよ処理の設計図に落とすのがこの節です。ここで押さえるのは二つの「順序」——**列ごとに処理を分ける**ことと、各列の中で**補完を先・変換を後**にすることです。

**なぜ列で分けるか**　数値の計測値と、文字のカテゴリ（`island`・`sex`）は、性質がまるで違います。計測値はものさしの数字なので「中央値で埋めて、標準化して土俵を揃える」。カテゴリはあてはまるか否かの区別なので「最頻値で埋めて、ワンホットで 0/1 の列に開く」。数値をワンホットにしても、文字を標準化しても意味をなしません。だから列の種類ごとに別の小さな流れ（Pipeline）を用意し、それを `ColumnTransformer` で**どの列にどちらを当てるか**の対応表として束ねます（第4回・第9回）。

**なぜ「補完 → 変換」の順か**　各列の中の順序にも理由があります。標準化は平均とばらつきを計算しますが、欠損（NaN）が混じったままでは計算が濁ります。ワンホットも、空欄をどう扱うか迷います。だから**先に穴を埋めてから**、標準化やワンホットに渡す。この順序を、小さな Pipeline `[('imp', ...), ('sc'/'oh', ...)]` が構造で保証します。

**設計図が生む列数（骨子）**　ワンホットは、カテゴリの取りうる値ごとに 0/1 の列を作ります。今回のデータでは `island` が3種類（Biscoe・Dream・Torgersen）、`sex` が2種類（Male・Female）。だから前処理を通したあとの特徴は、**数値4 + 島3 + 性別2 = 9列**になります（TODO⑥で `get_feature_names_out()` を見ると、まさに 9 個の名前が並びます）。生の6列が、モデルの食べられる 9 列の数値表に化けるわけです。

**なぜ `handle_unknown='ignore'` か**　ワンホットには「未知のカテゴリが来ても止まらない」指定を添えます。訓練で見なかった値がテストや本番で現れても、エラーで止まらず全ゼロで受け流すためです。これは次節のリーク防止と同じ思想——**訓練で決めた変換のルールを、あとから来るデータに機械的に当てる**——の一部です。

**この段が「ここ」に来る意味**　前処理の設計は、前節のデータ理解（列の型・欠損）を入力とし、次節の学習（この 9 列をモデルに渡す）へ出力します。ただし設計しただけでは、まだ実行しません。**いつ・何から学習させるか**は、次の Pipeline が引き受けます。


In [ ]:
# TODO: 数値用の前処理（補完→標準化）と、カテゴリ用の前処理（補完→ワンホット）をそれぞれ作り、

# 解答例②：ColumnTransformer で列ごとの前処理を束ねる
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('oh', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', num_pipe, num),
                          ('cat', cat_pipe, cat)])
print('前処理の設計図を組みました')
# → 数値は中央値補完＋標準化、カテゴリは最頻値補完＋ワンホット。列ごとに別の流れを当てている


## 3. モデルまで一本化する——Pipeline

前処理とモデルを、第9回の `Pipeline` で一本につなぎます。こうすれば、前処理→モデルの順序が構造で守られ、リークが起きません。中身のモデルは、第8回のロジスティック回帰です。

### TODO③：前処理とロジスティック回帰を、一つの Pipeline に結合する

### 深掘り：なぜ Pipeline で一本化するのか——リーク防止と、結果を再現する仕掛け

前節で前処理の**設計図**は組めました。でも設計図はまだ「いつ・何から学ぶか」を語っていません。それを構造で縛るのが、この節の Pipeline です。前処理とモデルを `[('prep', ...), ('model', ...)]` と一本につなぐと、`fit` を呼んだとき **前処理がまず訓練データだけから学習され、その学習済みのルールでモデルまで通る**、という順序が自動で守られます。

**なぜ順序を構造で縛るのか——リーク**　もし前処理を手作業で先に全データへ `fit` してしまうと、テストの平均やばらつき、最頻カテゴリが訓練の前処理に染み込みます。これが**リーク（答えの先読み）**で、成績が実力以上に見えてしまう。第3回で「テストは最後まで触らない」と誓った鉄則を、Pipeline は**うっかり破れないように**します。とくに交差検証（次節）では分割のたびに前処理を学び直す必要があり、これを手で正しくやるのはほぼ不可能です。Pipeline に入れておけば、各分割で自動的に「その分割の訓練部分からだけ」前処理が学ばれます。忙しいときの人為ミスを、そもそも起こせなくする——これが一本化のいちばんの利点です。

**発展：結果を再現する仕掛け（seed とバージョン固定）**　この演習を誰が何度実行しても同じ数字が出るのは、偶然ではありません。訓練とテストの分け方は `train_test_split(..., random_state=42)` の **乱数の種**で固定されています。種を止めているから、テストに入る 69 匹の顔ぶれが毎回同じで、あとの正解率も混同行列もぴたりと再現します。もし種を変えれば、テストの中身が入れ替わり、たとえば後で見る「たった1件の取り違え」が別の1匹に移ったり、消えたりし得ます。**再現性は、乱数の種を書き留めることから始まる**のです。

種だけでは足りません。ライブラリの**版**も効きます。この環境は numpy 2.2.6・pandas 2.3.3・scikit-learn 1.7.2。将来バージョンで既定値やアルゴリズムが変われば、同じコードでも数字が微妙にずれ得ます。だから実務では「誰が・いつ・どの版で・どの種で」出した結果かを記録に残す（環境を固定する）。分析は一度出して終わりではなく、**あとから同じ結果を再現できて初めて信頼される**——この習慣が、フローの土台を静かに支えています。


In [ ]:
# TODO: TODO② で作った前処理と、ロジスティック回帰を、一つの Pipeline にまとめてください

# 解答例③：前処理とモデルを Pipeline に結合
pipe = Pipeline([('prep', prep),
                 ('model', LogisticRegression(max_iter=5000))])
print('組み立てたステップ:', [name for name, _ in pipe.steps])
# → 'prep' → 'model' の順。fit を呼べば、前処理がまず訓練データから学習され、その後モデルが学習する


## 4. 実力を測る——交差検証

テストを見る前に、**訓練データの中だけ**で実力を測ります（第5回）。5分割の交差検証で正解率を出し、平均を確かめましょう。

### TODO④：Pipeline を交差検証で評価する

### 深掘り：なぜテストの前に交差検証を挟むのか——評価の順序と「指標の取り違え」

Pipeline が組めたら、すぐテストで採点したくなります。でもその前に、**訓練データの中だけ**で実力を測るのがこの節です（第5回）。なぜ一段はさむのか。テストは**最後の一発勝負**で、何度も見れば「テストに合わせた調整」が始まり、リークします。だから練習試合として、訓練を5分割し、4つで学び1つで測る——を5回まわして平均を取る**交差検証**を先に置きます。

TODO④の交差検証では、各分割の正解率が **[1.00, 0.98, 1.00, 0.98, 1.00]**、平均 **0.9927**（約99.3%）と出ます。テストにまだ一切触れずに、これだけの手応えが訓練の中だけで見えている——この段があるから、次節でテストを開く前に「たぶん高精度だ」と見当をつけ、しかもテストを汚さずに済みます。**評価は、内側（訓練の交差検証）→ 外側（テスト）の順**でしか安全に進められません。

**分割のばらつきも読む**　5つの正解率 [1.00, 0.98, 1.00, 0.98, 1.00] は、どれも 1.00 前後で互いに近い。この**ばらつきの小ささ**も大切な情報です。もし分割ごとに 0.7 と 1.0 が入り混じるなら、平均が高くても「たまたま良い分割に当たっただけ」かもしれず、実力を信じきれません。近い値が並ぶのは、どの 4/5 で学んでも安定して当てられる——**データの構造がはっきりしている**証拠です。1回の分割だけでは運不運が乗るので、5回に散らして平均とばらつきの両方を見る。こうして、一発勝負のテストへ進む前の見立てを堅くしておくわけです。

**つまずき：指標の取り違え**　交差検証には `scoring='accuracy'`（正解率）を指定しています。ここで立ち止まる価値があります——**その指標は、目標と噛み合っているか**。今日の目標は「種類を当てたい」。3種を等しく当てたいのだから、全体で何割当たったかの正解率が素直に目標を映します。しかも種の偏りは Adelie 152・Gentoo 124・Chinstrap 68 とそこそこ均されているので、正解率が偏りにだまされにくい（第8回で見た「全部を多数派と答えるだけで高い正解率」の罠が、ここでは効きにくい）。だから今回は正解率でよい。

でもこれは**目標しだい**です。もし目標が「Chinstrap だけは絶対に見逃したくない」なら、測るべきは Chinstrap の**再現率**であって、全体の正解率ではありません（第8回）。目標が変われば指標も変える——正解率をいつも既定で使うのは、分類でありがちなつまずきです。**指標は、目標を数字に翻訳したもの**。翻訳を間違えれば、高い点数が出ても「そもそも測りたいものを測っていない」ことになります。だから評価の段では、必ず「この指標は今日の目標を映しているか」を一度問い直してください。


In [ ]:
# TODO: 組み立てた Pipeline を、訓練データに対して交差検証で評価し、各分割のスコアと平均を確認してください

# 解答例④：交差検証で実力を測る
cv = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring='accuracy')
print('各分割の正解率:', np.round(cv, 3))
print(f'平均: {cv.mean():.3f}')
# → どの分割も非常に高い。テストを見る前に、訓練データの中だけで健全に測れている


## 5. 最終評価と、間違いの中身——混同行列

`GridSearchCV` で正則化の強さ `C` を選び（第9回）、選ばれた Pipeline をテストで**一度だけ**評価します。そして、正解率だけでは見えない**間違いの内訳**を、混同行列で確かめます。まずは探索とテスト評価を済ませ、その予測から混同行列を作ってください。

### TODO⑤：C を探索してテスト評価し、混同行列を作る

### 深掘り：なぜテストは一度だけか／実務でのモデル選択基準

いよいよテストを開きますが、その前に一仕事あります——正則化の強さ `C` を選ぶことです。ここで大事なのは、**選ぶ作業もテストを使わずに行う**こと。`GridSearchCV` は候補 `C ∈ {0.01, 0.1, 1, 10, 100}` を、訓練データの中の交差検証で総当たりし、いちばん成績の良い設定を選びます。テストは、選び終わったモデルを**最後に一度だけ**採点するために取ってあります。もし `C` 選びにテストを使えば、テストは「未知の本番」ではなくなり、リークします。だからテストは一発勝負——**一度見たら、それはもう本番ではない**のです。

**モデル選択基準を、実務ではどう決めるか（発展）**　`GridSearchCV` が並べる各 `C` の平均交差検証スコアは、実測で次のようになります。

| `C` | 0.01 | 0.1 | 1 | 10 | 100 |
|---:|---:|---:|---:|---:|---:|
| CV正解率 | 0.924 | 0.982 | 0.993 | **0.996** | 0.996 |

いちばん強い正則化（`C`=0.01）は 0.924 と明らかに沈み、ゆるめるほど上がって `C`=10 で頭打ちになります。選ばれたのは `C`=10（CV 0.9964）。ここに実務の勘どころが二つあります。**一つ目、端が選ばれていないか**。もし候補の端（0.01 や 100）が選ばれたら、探索範囲が狭すぎるサインで、もっと外側も試すべきです。今回は内側の 10 が選ばれたので、この範囲で山の頂を捉えられたと判断できます。**二つ目、僅差なら簡素な側を採る**。`C`=10 と 100 はどちらも 0.9964 と同点です。同じ成績なら、より強く正則化した（＝より単純な）`C`=10 を採るのが穏当——これは私たちが下す**人の判断**です。念のため補足すると、`GridSearchCV` は「単純なほうを好む」わけではありません。同点のときは `cv_results_` に**先に現れた候補**、つまり `param_grid` に並べた順で先に来たほうを採るだけで、今回 `C`=10 が選ばれたのは候補を昇順（0.01→…→100）に並べた結果の偶然です。もし降順に並べれば、同じ同点でも `C`=100 が選ばれます。だから「僅差なら簡素な側」はライブラリ任せにせず、こちらの基準として意識して選ぶべきものです。「性能が実質同じなら単純なモデルを選ぶ」——この考えを一歩進めた**1標準誤差ルール**（最良から誤差1つ分以内でいちばん単純な設定を選ぶ）もありますが、深入りはしません。

**なぜ選択用の交差検証を、そのまま成績にできないのか**　ここで一つ、見落としやすい点があります。最良スコア 0.9964 は、**5つの候補の中からいちばん高いものを選んだ**値です。たくさん試して最大を拾えば、選ばれた値は本当の実力より少し上ぶれしがち（選択のバイアス）。だから、選ぶのに使った交差検証の点数を、そのまま「本番の成績」と名乗らせるわけにはいきません。選択に一切関わっていない**別腹のテスト**が要る——これが、選択用のCVとテストを分けて取り分けておく理由です。より厳密には、選択の交差検証をもう一段の交差検証で包む**入れ子交差検証（nested CV）**という手もありますが、ここでは「選ぶための評価と、実力を測るための評価は分ける」という原則を押さえれば十分です。

**テストで答え合わせ**　選ばれた `C`=10 のモデルをテストで採点すると、正解率 **0.9855**（約98.5%）。交差検証の 0.9964 とほぼ一致します。この**内側と外側が近い**ことこそ、設定選びが健全だった証拠です（もしテストがぐっと低ければ、訓練に合わせすぎたサイン）。そして正解率だけで終えず、混同行列で中身を見ます。テスト 69 匹の誤りは**わずか1件、Adelie を Chinstrap と取り違えた**だけで、Chinstrap と Gentoo は取りこぼしゼロ（再現率1.00）でした。正解率という一つの数字の裏に「どの種どうしが紛れたか」が隠れており、それは次節で最初の散布図と符合します。


In [ ]:
# TODO: C の候補をいくつか用意して GridSearchCV で探索し、選ばれた Pipeline をテストで評価してください。

# 解答例⑤：GridSearchCV で C を選び、テスト評価と混同行列
param_grid = {'model__C': [0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
gs.fit(X_tr, y_tr)
print('最良の設定:', gs.best_params_)
print(f'交差検証での正解率: {gs.best_score_:.4f}')
print(f'テストでの正解率:   {gs.score(X_te, y_te):.4f}')

labels = ['Adelie', 'Chinstrap', 'Gentoo']
cm = confusion_matrix(y_te, gs.predict(X_te), labels=labels)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel('予測'); ax.set_ylabel('正解')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else '#333', fontsize=14)
plt.tight_layout(); plt.show()
print(classification_report(y_te, gs.predict(X_te)))
# → 選ばれた C は内側の値。テスト正解率は交差検証とほぼ一致。誤りは Adelie と Chinstrap の間にわずか、Gentoo は完璧


## 6. 目標との整合を確かめる——係数と散布図

最後に、③の解釈です。**どの特徴が効いたか**を係数で見て、それを**最初の散布図に戻って**確かめます。数字と図が一致すれば、結果を心から納得できます。

### TODO⑥：効いた特徴を係数で調べ、くちばし長×深さの散布図で裏づける

### 深掘り：なぜ最後が「解釈と報告」か——目標に立ち返り、数字を図で裏づける

数字は出そろいました。でも分析は、正解率を出して終わりではありません。**最後に目標へ立ち返り、結果を人に伝わる形にする**——この段があって初めて一周が閉じます。だから解釈と報告が末尾に来ます。ここを省くと「98.5%当たった」で止まり、**何が効いたのか・信じてよいのか**が宙に浮きます。

**数字を図に戻す**　TODO⑥で係数を見ると、Adelie らしさにいちばん効くのは `bill_length_mm`（くちばしの長さ）で、係数は **−4.43** と大きな負。「くちばしが**短い**ほど Adelie らしい」という意味です。ここで大事なのは、この数字を**最初の散布図に戻って**確かめること。散布図の上で Adelie が左（短いくちばし側）に固まっていれば、係数の話と図が一致し、心から納得できます。**モデルが賢いのではなく、データにもともとそういう構造があった**——解釈とは、その構造を数字と図の往復で腑に落とす作業です。第6回の「係数を読む」型が、ここでそのまま効いています。

**つまずき：目標を見失った分析**　フローを回すうちに、いつのまにか「正解率をあと0.1%上げる」こと自体が目的化する——これがいちばん多い落とし穴です。今日の目標は「くちばしと体の計測値から種類を当てたい」でした。98.5%はその目標に十分答えています。ここから正則化やモデルをこねくり回して小数点以下を追うのは、**目標を見失った分析**です。逆に、目標に照らせば**まだ答えられないこと**もはっきり言えます——誤りは似た Adelie と Chinstrap の間に残る（散布図で近い種どうし）、そしてこのデータはパーマー基地の3種のみで、**他の場所・他の種にそのまま使える保証はない**。何ができて何ができないかを、目標を軸に線引きするのが解釈の仕事です。

**限界も、結果の一部として報告する**　良い報告は、できたことだけを並べません。「Adelie と Chinstrap の間に誤りが残る」「パーマー基地の3種にしか確かめていない」といった**限界**まで書いて、初めて誠実な報告になります。聞き手が次に何を疑い、何を追加で確かめるべきかは、この限界の記述から決まるからです。都合の悪い数字を隠さない——これも、目標に正直であり続けるということです。

**報告の三点セット（第6回）**　最後に結果を、**目的・方法・結果**の三つでまとめます。目的＝4計測値と島・性別から3種を当てる。方法＝中央値/最頻値で補完 → 標準化/ワンホット → ロジスティック回帰、交差検証で `C` を選びテストで評価。結果＝テスト正解率 約98.5%、くちばしの長さが最も効き、誤りは Adelie と Chinstrap の間に1件。この順に並べるだけで、話がすっと通ります。**分析は、答えを出し、それを人に伝わる形にまとめるまでが仕事**。ここまでたどって、目標設定に始まった一周が、ようやく閉じます。次回からは正解（ラベル）のない世界——後半戦へ進みます。


In [ ]:
# TODO: 学習済みモデルの係数から、Adelie らしさに効く特徴を大きい順に調べ、

# 解答例⑥：効いた特徴を係数で調べ、散布図で裏づける
best = gs.best_estimator_
names = best.named_steps['prep'].get_feature_names_out()
coef = best.named_steps['model'].coef_[0]  # Adelie らしさへの寄与
top = np.argsort(np.abs(coef))[::-1][:4]
print('Adelie らしさに効く特徴（上位4）:')
for j in top:
    print(f'  {names[j]:24s} {coef[j]:+.2f}')

fig, ax = plt.subplots(figsize=(8, 4.4))
for sp, c in zip(labels, ['#0066cc', '#2a9d8f', '#e63946']):
    d = df[df['species'] == sp]
    ax.scatter(d['bill_length_mm'], d['bill_depth_mm'], s=18, alpha=0.7, color=c, label=sp)
ax.set_xlabel('くちばしの長さ (mm)'); ax.set_ylabel('くちばしの深さ (mm)')
ax.legend()
plt.tight_layout(); plt.show()
# → いちばん効くのは bill_length_mm（大きな負の係数）。散布図でも Adelie は左（くちばしが短い側）に固まる。数字と図が一致


## 目標に答えられたか

- 今日の目標は「くちばしと体の計測値から、ペンギンの種類を当てたい」でした
- TODO④の交差検証と、TODO⑤のテスト正解率は、近い値でしたか。近ければ、設定選びは健全だったと言えます
- TODO⑤の混同行列で、**どの種どうし**の取り違えが残りましたか。それは、最初の散布図で近くにいた種でしたか
- TODO⑥で、いちばん効いた特徴は何でしたか。その特徴で、散布図の上でも種が分かれて見えましたか（係数と図の一致）
- **まだ答えられないこと**：このデータはパーマー基地の3種だけ。他の場所・他の種にそのまま使えるかは分かりません
- ここまでで、①理解 → ②前処理・解析 → ③解釈を、一本の流れで回せました。次回からは後半戦——ラベルのない世界へ進みます


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

この課題は、基本の TODO②〜⑥を終えてから取り組んでください。

TODO⑥で、Adelie らしさにいちばん効いた特徴は `bill_length_mm`（くちばしの長さ）でした。では、その1列が**測れなかった**らどうなるでしょう。
数値の列を `bill_depth_mm`・`flipper_length_mm`・`body_mass_g` の3列だけにして（`island`・`sex` はそのまま使います）、TODO②→③→⑤と同じ手順で `GridSearchCV` まで回し、テスト正解率と混同行列（3×3）を表示してください。


In [ ]:
# くちばしの長さを外した3列で、TODO②→③→⑤と同じ手順をやり直す
num3 = ['bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
prep3 = ColumnTransformer([('num', num_pipe, num3),
                           ('cat', cat_pipe, cat)])
pipe3 = Pipeline([('prep', prep3),
                  ('model', LogisticRegression(max_iter=5000))])
gs3 = GridSearchCV(pipe3, param_grid, cv=5, scoring='accuracy')
gs3.fit(X_tr, y_tr)
print('最良の設定:', gs3.best_params_)
print(f'交差検証での正解率: {gs3.best_score_:.4f}')
print(f'テストでの正解率:   {gs3.score(X_te, y_te):.4f}')

cm3 = confusion_matrix(y_te, gs3.predict(X_te), labels=labels)
print(pd.DataFrame(cm3, index=[f'正解:{l}' for l in labels],
                   columns=[f'予測:{l}' for l in labels]))


<details><summary>詰まったら</summary>

変えるのは `ColumnTransformer` に渡す数値列のリストだけです。`num` の代わりに `bill_length_mm` を除いた3列のリストを渡し、あとは TODO③⑤のコードをそのまま実行します。`X_tr` から列を消す必要はありません（`ColumnTransformer` は指定した列だけを拾い、残りは捨てます）。

</details>


### 応用②（判断）

応用①の、くちばしの長さを外したモデルの**テスト正解率**を、**小数第2位まで**答えてください（例: 0.91）。


In [ ]:
acc3 = gs3.score(X_te, y_te)
print('答え:', round(acc3, 2))


<details><summary>詰まったら</summary>

TODO⑤と同じく `GridSearchCV` の `.score(X_te, y_te)` で出ます。`round(値, 2)` で丸めてください。

</details>


### 応用③（解釈）

応用①の混同行列で、最も多く取り違えられた種の組はどれですか。くちばしの長さを外すとなぜその2種が紛れるのかを、6節の散布図（横軸がくちばしの長さ）と関連づけて、パーマー基地の調査担当者に向けて3行で書いてください。


**模範例**

最も多く取り違えられたのは Adelie と Chinstrap の組で、テスト 69 羽のうち Adelie 6 羽が Chinstrap に、Chinstrap 5 羽が Adelie に判定されました（Gentoo は 25 羽すべて正解、テスト正解率 0.84）。

この2種はくちばしの深さ・翼の長さ・体重がほぼ同じで、6節の散布図でも横軸のくちばしの長さだけで左右に分かれていたので、その横軸を外すと2種を分ける手がかりがほとんど残りません。

Gentoo はくちばしが浅く体が重いので、長さがなくても区別できます。

現場では、くちばしの長さを必ず測ることが、Adelie と Chinstrap の判定精度を保つ鍵になります。


## 発展（任意）

### 学習済みモデルを保存して、新しい1羽を判定する

分析は「作って終わり」ではありません。来シーズンに新しい個体が測られたら、同じモデルで種を判定したくなります。
そのたびに学習をやり直すのは手間ですし、データや版がずれれば結果もぶれます。学習済みの Pipeline を**ファイルに保存**しておけば、読み込むだけで同じ判定が再現できます。

ここで Pipeline の利点がもう一つ効きます。補完・標準化・ワンホットの**ルールごと**保存されるので、新しいデータに手作業の前処理は要りません。訓練のときと同じ列名の表を渡すだけです。

保存には公式文書でも紹介されている `joblib` を使います。下のセルは本文の変数に頼らず、TODO②〜⑤と同じ Pipeline（`C`=10）を組み直してから保存します。


In [ ]:
import os
import joblib

# TODO②〜⑤と同じ Pipeline を組み直し、訓練データで学習する（C は TODO⑤で選ばれた 10）
num_pipe_adv = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('sc', StandardScaler())])
cat_pipe_adv = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                         ('oh', OneHotEncoder(handle_unknown='ignore'))])
prep_adv = ColumnTransformer([('num', num_pipe_adv, num),
                              ('cat', cat_pipe_adv, cat)])
model_adv = Pipeline([('prep', prep_adv),
                      ('model', LogisticRegression(C=10, max_iter=5000))]).fit(X_tr, y_tr)

# 1) ファイルに保存する
joblib.dump(model_adv, 'penguin_model.joblib')
print('保存したファイルの大きさ:', os.path.getsize('penguin_model.joblib'), 'バイト')

# 2) 読み込む（別の日・別のノートで、学習なしにここから始められる）
loaded = joblib.load('penguin_model.joblib')
print('読み込んだステップ:', [name for name, _ in loaded.steps])


In [ ]:
# 3) 新しく測った3羽を、訓練のときと同じ列名で表にして判定する（2羽目は性別が未記入）
new_birds = pd.DataFrame([
    {'bill_length_mm': 45.0, 'bill_depth_mm': 18.5, 'flipper_length_mm': 195.0,
     'body_mass_g': 3800.0, 'island': 'Dream', 'sex': 'Male'},
    {'bill_length_mm': 39.0, 'bill_depth_mm': 18.5, 'flipper_length_mm': 190.0,
     'body_mass_g': 3700.0, 'island': 'Torgersen', 'sex': None},
    {'bill_length_mm': 47.0, 'bill_depth_mm': 15.0, 'flipper_length_mm': 215.0,
     'body_mass_g': 5000.0, 'island': 'Biscoe', 'sex': 'Female'},
])
proba = pd.DataFrame(loaded.predict_proba(new_birds), columns=loaded.classes_).round(3)
proba['判定'] = loaded.predict(new_birds)
display(pd.concat([new_birds, proba], axis=1))

# 後片付け（この演習ではファイルを残さない。実務では残して使い回す）
os.remove('penguin_model.joblib')


**読み方**　保存したファイルは約 5 KB（この環境では 4826 バイト）です。前処理からモデルまでの一式が、この小さなファイルに収まっています。読み込んだ `loaded` は `'prep'` → `'model'` の2ステップを持ち、学習前の準備なしにそのまま判定に使えます。

3羽の判定を見ます。1羽目（くちばし 45.0 mm・Dream 島）は **Chinstrap 0.633、Adelie 0.359** と、判定は Chinstrap ですが確信は6割ほどです。くちばしの長さ 45 mm は Adelie（平均 38.8 mm）と Chinstrap（平均 48.8 mm）のほぼ中間で、6節の散布図でも2種が重なる帯にいます。`predict` の答えだけでなく `predict_proba` の確率を見ると、この個体は「要確認」として扱うのが安全です。

2羽目は性別が未記入（`None`）のままでも止まりません。保存された補完器が最頻値で埋めてから判定し、**Adelie 1.000** と迷いなく答えます。3羽目は体重 5000 g・くちばしの深さ 15 mm で、**Gentoo 0.999** です。

注意点を一つ。joblib のファイルは、保存したときの scikit-learn の版に結びついています。別の版で読み込むと警告が出て、動作が保証されません。3節の深掘りで見た「版を記録する」習慣は、モデルを配るときにこそ効きます。

試すなら、1羽目の `bill_length_mm` を 42 や 48 に変えて、Adelie と Chinstrap の確率がどこで入れ替わるかを見てください。
